# Heart Disease ML (Runnable)
This notebook uses only standard scikit-learn components.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, matthews_corrcoef,
    roc_auc_score, average_precision_score,
    balanced_accuracy_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay
)


In [ ]:
url="https://raw.githubusercontent.com/btenneson/public_projects/refs/heads/main/predator/heart_disease_health_indicators_BRFSS2015.csv"

df=pd.read_csv(url)
X=df.iloc[:,1:]
y=df.iloc[:,0]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

pipe=Pipeline([
    ("imputer",SimpleImputer()),
    ("model",GradientBoostingClassifier(random_state=42))
])

grid={
    "model__n_estimators":[100,200],
    "model__learning_rate":[0.05,0.1],
    "model__max_depth":[2,3]
}

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

search=GridSearchCV(
    pipe,
    grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

search.fit(X_train,y_train)
best=search.best_estimator_
print(search.best_params_)


In [ ]:
train_prob=best.predict_proba(X_train)[:,1]

thresholds=np.linspace(0.1,0.9,161)
best_threshold=0.5
best_mcc=-1

for t in thresholds:
    pred=(train_prob>=t).astype(int)
    mcc=matthews_corrcoef(y_train,pred)
    if mcc>best_mcc:
        best_mcc=mcc
        best_threshold=t

print(best_threshold,best_mcc)


In [ ]:
prob=best.predict_proba(X_test)[:,1]
pred=(prob>=best_threshold).astype(int)

print(classification_report(y_test,pred))
print("ROC-AUC:",roc_auc_score(y_test,prob))
print("PR-AUC:",average_precision_score(y_test,prob))
print("Balanced Accuracy:",balanced_accuracy_score(y_test,pred))
print("MCC:",matthews_corrcoef(y_test,pred))

RocCurveDisplay.from_predictions(y_test,prob)
plt.show()

PrecisionRecallDisplay.from_predictions(y_test,prob)
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()
